# Lezione 50 — RLHF, RLAIF e DPO a confronto

> **Modulo:** Feedback e preference training · **Tempo stimato:** 25 minuti
> **Prerequisiti:** Lezione 47 (reward), Lezione 49 (preference tuning).
>
> Obiettivo pratico unico: capire le tre famiglie di allineamento da preferenze
> e mostrare in NumPy come un **reward model faccia da etichettatore** (RLAIF) al
> posto dell'umano.

## Teoria essenziale

Tre approcci per allineare un modello alle preferenze:

- **RLHF** (Reinforcement Learning from **Human** Feedback; Christiano et al.,
  2017; Ouyang et al., 2022): umani producono le preferenze → si addestra un
  reward model → si ottimizza la politica con RL (con vincolo KL).
- **RLAIF** (from **AI** Feedback; Bai et al., 2022; Lee et al., 2023): le
  preferenze le genera un **modello** (un reward model o un LLM giudice) invece
  degli umani — piu' scalabile ed economico, ma eredita i **bias** del giudice.
- **DPO** (Rafailov et al., 2023): salta reward model + RL e ottimizza la
  politica direttamente dalle coppie (Lezioni 48–49). Puo' usare coppie umane
  (→ stile RLHF) o generate da AI (→ stile RLAIF).

Il punto chiave: RLHF/RLAIF differiscono per **chi etichetta**; DPO differisce
per **come si ottimizza**. Sono assi ortogonali.

In [ ]:
import numpy as np

rng = np.random.default_rng(50)

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

# preferenza VERA (umana ideale) definita da theta_umano
d = 5
theta_umano = rng.normal(size=d)

# un reward model AI (imperfetto): theta_ai = umano + rumore. Piu' rumore = giudice peggiore
def fai_giudice_ai(rumore):
    return theta_umano + rng.normal(size=d) * rumore

# coppie da etichettare
N = 400
Fa = rng.normal(size=(N, d))
Fb = rng.normal(size=(N, d))
# etichetta 'umana' (con un po' di rumore realistico)
umano_a_meglio = rng.random(N) < sigmoid((Fa - Fb) @ theta_umano)
print("coppie:", N, "| dim:", d)

In [ ]:
# RLAIF: il giudice AI etichetta le coppie. Quanto concorda con l'umano?
for rumore in [0.0, 0.5, 1.0, 2.0]:
    theta_ai = fai_giudice_ai(rumore)
    ai_a_meglio = (Fa - Fb) @ theta_ai > 0
    accordo = float((ai_a_meglio == umano_a_meglio).mean())
    print(f"giudice AI con rumore {rumore:.1f}: accordo con l'umano = {accordo:.2f}")

Leggi l'accordo: un giudice AI *buono* (poco rumore) etichetta quasi come
l'umano → RLAIF e' quasi gratis e scalabile. Un giudice *scadente* (molto rumore)
si allontana: RLAIF eredita i limiti del suo etichettatore. E' il compromesso
centrale: scala vs fedelta'.

In [ ]:
# effetto a valle: una politica DPO addestrata su etichette AI buone vs scadenti
def tuning_da_etichette(theta_giudice, passi=200, lr=0.5):
    a_meglio = (Fa - Fb) @ theta_giudice > 0
    Fw = np.where(a_meglio[:, None], Fa, Fb)
    Fl = np.where(a_meglio[:, None], Fb, Fa)
    th = np.zeros(d)
    for _ in range(passi):
        p = sigmoid((Fw - Fl) @ th)
        th -= lr * (-((1 - p)[:, None] * (Fw - Fl)).mean(axis=0))
    # qualita' = allineamento con la preferenza UMANA vera
    return float(th @ theta_umano / (np.linalg.norm(th) * np.linalg.norm(theta_umano)))

buono = tuning_da_etichette(fai_giudice_ai(0.3))
scadente = tuning_da_etichette(fai_giudice_ai(2.0))
print(f"politica DPO da giudice AI buono   -> allineamento umano {buono:.3f}")
print(f"politica DPO da giudice AI scadente -> allineamento umano {scadente:.3f}")
assert buono > scadente
print("conferma: la qualita' del giudice AI si propaga alla politica finale")

## Il progetto: Memory AI Lab, passo 50

Nel progetto, un reward model addestrato (Lezione 47) puo' fare da giudice AI per
etichettare nuove coppie di memorie a basso costo (RLAIF) — utile quando il
feedback umano scarseggia — a patto di **monitorarne la qualita'** contro un
piccolo campione umano.

## Riepilogo (max 8 punti)

1. **RLHF**: preferenze umane → reward model → RL con vincolo KL.
2. **RLAIF**: le preferenze le genera un modello (scala, ma eredita bias).
3. **DPO**: ottimizza la politica direttamente dalle coppie (no RL).
4. RLHF vs RLAIF = **chi etichetta**; DPO = **come si ottimizza** (assi
   ortogonali).
5. Un giudice AI buono concorda quasi come l'umano.
6. Un giudice scadente si allontana: RLAIF eredita i suoi limiti.
7. La qualita' del giudice si **propaga** alla politica finale.
8. RLAIF va sempre monitorato contro un campione umano.

## Quiz

1. Qual e' la differenza fondamentale tra RLHF e RLAIF?
2. DPO e' alternativo o compatibile con etichette AI?
3. Perche' RLAIF va monitorato contro giudizi umani?

*(Risposte: 1. chi produce le preferenze — umani vs un modello; 2. compatibile:
DPO puo' usare coppie da qualunque etichettatore; 3. perche' eredita i bias del
giudice AI, che vanno controllati contro la verita' umana.)*